In [ ]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict

from Utils.Config import Config, TrialResult
from Utils.utils import (
    set_seed,
    make_surv_y,
    make_oof_predictions,
    create_features,
    load_config_yaml,
    HORIZONS
)

In [2]:
storage = JournalStorage(
    JournalFileBackend("gbsa_journal.log")
)

study = optuna.load_study(
    study_name="gbsa_survival",
    storage=storage,
)

In [3]:
best_trial = study.best_trial
best_trial_num = best_trial.number
print("Best trial:")
print(f"  Number: {best_trial.number}")
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Number: 106
  Value: 0.9629674137093398
  Params: 
    n_estimators: 417
    learning_rate: 0.11063731185373184
    subsample: 0.5905037758154414
    max_depth: 6
    max_features: sqrt
    max_leaf_nodes: 31
    min_samples_split: 4
    min_samples_leaf: 8
    min_impurity_decrease: 0.02904783931441647
    ccp_alpha: 0.004992133979210339
    dropout_rate: 0.0007471276215608402
    validation_fraction: 0.1897124693474042
    n_iter_no_change: None
    tol: 0.0004912572712305443


In [4]:
trial_dir = os.path.join("Trials", "GBSA")
best_trial_path = os.path.join(trial_dir, f'trial_{best_trial.number}', 'config.yaml')
print(f"Best trial config path: {best_trial_path}")
print(f"Best trial config exists: {os.path.exists(best_trial_path)}")

Best trial config path: Trials/GBSA/trial_106/config.yaml
Best trial config exists: True


In [5]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [6]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


In [7]:
config = load_config_yaml(best_trial_path)
print("Loaded config:")
print(config)

Loaded config:
Config(seed=42, cv_n_splits=5, cv_n_repeats=10, gbsa_config=GBSAConfig(loss='coxph', n_estimators=417, learning_rate=0.11063731185373184, subsample=0.5905037758154414, max_depth=6, max_features='sqrt', max_leaf_nodes=31, min_samples_split=4, min_samples_leaf=8, min_weight_fraction_leaf=0.0, min_impurity_decrease=0.02904783931441647, criterion='friedman_mse', ccp_alpha=0.004992133979210339, dropout_rate=0.0007471276215608402, validation_fraction=0.1897124693474042, n_iter_no_change=None, tol=0.0004912572712305443, random_state=42, warm_start=False, verbose=0), preprocessing_config=PreprocessingConfig(eps=1e-06, min_speed=0.01, max_hours=9999.0))


In [ ]:
seed = config.seed
set_seed(seed)
model_params = config.gbsa_config

n_splits = config.cv_n_splits
n_repeats = config.cv_n_repeats

print(n_splits, n_repeats)

In [9]:
model = GradientBoostingSurvivalAnalysis(**asdict(model_params))

oof_pred = make_oof_predictions(
    model=model,
    data=train_processed,
    horizons=event_horizon,
    seed=seed,
    n_splits=n_splits,
    n_repeats=n_repeats
)
print(oof_pred)

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

{'final_oof_risk': array([-0.95422292,  2.80563705,  3.73880235, -1.55702051, -1.26366121,
       -1.47569768,  2.40444177, -1.13313123, -1.39665391,  1.65003136,
       -1.51454957,  2.85668923,  1.85835523, -1.52302872, -1.53202476,
       -1.48201131,  2.50957054,  2.44415547,  4.66897514, -1.53316197,
       -1.12338525, -1.52112214, -1.5117804 , -1.48538701,  4.52213161,
       -1.47907024,  4.05278267,  2.87509405, -1.5084563 , -1.53848902,
        4.41753685,  3.56094777,  2.19813216, -1.52441536, -1.51625807,
       -1.50933459,  2.62544227,  2.23949006, -1.50414651, -1.39688308,
       -1.4878827 , -1.17103745,  4.26306884,  2.89239985,  2.45192505,
       -1.33964277, -1.5050198 ,  2.01218285, -1.44109376, -1.52850725,
       -1.07047606, -1.50374923, -1.47162335,  1.88990033, -1.50391079,
       -1.4752087 , -1.47518445,  2.35098583, -1.5230746 , -1.50962991,
        3.85824031, -1.47618891,  3.37498335, -1.47994117, -1.50487353,
       -1.53822489, -1.49486375, -1.44036285,

In [ ]:
repeat_results = oof_pred['repeat_oof_results']
pred_matrix = []

for oof in repeat_results:
    pred_matrix.append(oof['preds'])

{'repeat': 1, 'oof_risk': array([-1.01090027,  2.61324777,  3.90163479, -1.50796426, -1.21353077,
       -1.58441861,  2.44762131, -1.00227818, -1.45280194,  1.82166068,
       -1.46130593,  2.66608561,  1.82560979, -1.50743298, -1.54158873,
       -1.34424831,  2.7918893 ,  2.3195092 ,  4.52144661, -1.63447224,
       -1.28454634, -1.53920987, -1.43930117, -1.50668798,  4.34773236,
       -1.34811451,  3.91948513,  3.10050183, -1.57568058, -1.61347436,
        4.59454472,  3.45443545,  2.15969728, -1.43930117, -1.57502271,
       -1.56718107,  2.92383054,  2.25518989, -1.57412577, -1.51325646,
       -1.51816506, -1.33647688,  4.1437305 ,  2.84851212,  2.32373125,
       -1.24214288, -1.44028963,  1.79126541, -1.57905171, -1.50668798,
       -0.99720837, -1.56465793, -1.53006052,  1.56664427, -1.54753586,
       -1.35493961, -1.55665099,  2.17654571, -1.5266631 , -1.51473225,
        3.60526801, -1.43906665,  3.64325063, -1.53006052, -1.50743298,
       -1.56870954, -1.33736241, -1.31